# TissueMC: does the depth-profile atomic set the run time?

R1, R5 and R6 all take the same time (about 10.06 s on 4 million photons) although they differ in local memory, in instruction count and in relaunch. R6's counters point at one thing they share: the atomic add into the depth profile that every absorption event makes. One L2 slice runs at 64% of peak against a 2% average, and R6's wait is 78% "load/store queue full" with atomics as its only load/store traffic.

This notebook tests that by **removing the depth and radius profile tallies from R6** (`--no-profiles`, a diagnostic: the profiles come out as zeros and nothing else changes) and timing it. If the time drops, the atomics are the floor; if it does not, the reading was wrong. **The diagnostic's numbers are not a result**; they say what the atomics cost.

It checks that the other tallies still agree with the CPU, benchmarks **R6, R6 without profiles, R1, then R6 again in the same session** (so the GPU's drift is visible), and collects every counter for R6 and R6 without profiles. It ends with a summary block and a digest for you to paste back.

**Before you start:** *Runtime → Change runtime type → T4 GPU.* Nothing else to record.

**Step 1 is a short check.** Every later step starts only if it passed; otherwise it prints "SKIPPED". If step 1 fails, **paste its output**. Raw counters are kept in `results/*.ncu.csv` and reused if you run a step again (a session that still has R6's from last time will reuse them). Expect about 20 minutes. Nothing here needs passwords or keys.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv,noheader
!nvcc --version | tail -1

In [ ]:
# Safe to run twice: reuse the clone if it is already there instead of nesting a second one.
!if [ -d /content/tissuemc/.git ]; then git -C /content/tissuemc pull -q; else git clone -q https://github.com/Raghava-Dwivedi/tissuemc.git /content/tissuemc; fi
%cd /content/tissuemc
!git log -1 --format='commit %h  %s'

In [ ]:
!pip install -q cmake ninja numpy matplotlib
!pip install -q -e .

In [ ]:
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DTISSUEMC_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null
!cmake --build build 2>&1 | tail -3
!./build/tissuemc --version

In [ ]:
# 1. Quick check: with the profiles off, do R_d, T_t, absorbed weight and the event count still agree
# with the CPU on identical photons? (The depth and radius profiles are not compared; they are
# zeros by design, and the check refuses a run that still reports them.) The later steps run only
# if this passes (it leaves results/.quick_ok).
!rm -f results/.quick_ok
!tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 6 --b-no-profiles --photons 200000 && touch results/.quick_ok

In [ ]:
# 2. The same comparison on 2,000,000 photons, kept as a record.
!if [ -f results/.quick_ok ]; then tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 6 --b-no-profiles --photons 2000000 --report results/gpu-r6-noprofiles-cross-engine.json; else echo "SKIPPED: the quick check did not pass, so the full comparison was not run"; fi

In [ ]:
# 3. Benchmarks back to back: R6, R6 without profiles, R1, then R6 again (drift). About 7 minutes.
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 6 --binary build/tissuemc --output results/gpu-r6-first.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the rung 6 benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 6 --no-profiles --binary build/tissuemc --output results/gpu-r6-noprofiles.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the diagnostic benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 1 --binary build/tissuemc --output results/gpu-r1-ablation-session.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the rung 1 benchmark was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-bench --engine cuda --rung 6 --binary build/tissuemc --output results/gpu-r6-again.json 2>&1 | tail -4; else echo "SKIPPED: the quick check did not pass, so the second rung 6 benchmark was not run"; fi

In [ ]:
# 4. Every Nsight Compute counter for R6 (reused if this session already has it) and for the
# diagnostic, 500,000 photons. About 3 to 6 minutes each.
!if [ -f results/.quick_ok ]; then tissuemc-profile --rung 6 --full --reuse-csv --photons 500000 --binary build/tissuemc --output results/gpu-r6-full.json; else echo "SKIPPED: the quick check did not pass, so the rung 6 profile was not run"; fi
!if [ -f results/.quick_ok ]; then tissuemc-profile --rung 6 --no-profiles --full --reuse-csv --photons 500000 --binary build/tissuemc --output results/gpu-r6-noprofiles-full.json; else echo "SKIPPED: the quick check did not pass, so the diagnostic profile was not run"; fi

In [ ]:
# 5. The digest: A is R6, B is R6 without the profile atomics. Paste this cell's output back.
!if [ -f results/gpu-r6-full.json ] && [ -f results/gpu-r6-noprofiles-full.json ]; then tissuemc-profile --digest results/gpu-r6-full.json results/gpu-r6-noprofiles-full.json; else echo "SKIPPED: a profile above did not finish"; fi

In [ ]:
import sys

sys.path.insert(0, "python")
import json
import pathlib
import subprocess


def load(name):
    p = pathlib.Path("results") / name
    return json.loads(p.read_text()) if p.exists() else None


def bench_summary(b):
    if not b:
        return None
    return {
        "median_kernel_s": round(b["kernel"]["median"], 3),
        "kernel_seconds": [round(x, 3) for x in b["kernel_seconds"]],
        "million_events_per_s": round(b["events_per_second"] / 1e6, 1),
        "validated_first": b["validated_before_measuring"],
        "profiles": b.get("profiles", True),
    }


def cross_summary(c):
    if not c:
        return None
    return {
        "agree": c["agree"],
        "gaps": {g["name"]: [g["value"], g["limit"]] for g in c["gaps"]},
    }


commit = subprocess.run(
    ["git", "log", "-1", "--format=%H"], capture_output=True, text=True
).stdout.strip()
noprofiles = load("gpu-r6-noprofiles.json")
summary = {
    "commit": commit,
    "device": noprofiles["device"] if noprofiles else None,
    "cross_engine_r6_noprofiles": cross_summary(load("gpu-r6-noprofiles-cross-engine.json")),
    "benchmark": {
        "r6_first": bench_summary(load("gpu-r6-first.json")),
        "r6_noprofiles": bench_summary(noprofiles),
        "r1": bench_summary(load("gpu-r1-ablation-session.json")),
        "r6_again": bench_summary(load("gpu-r6-again.json")),
    },
}
print("===== ABLATION SUMMARY =====")
print(json.dumps(summary, indent=1))